In [1]:
import re
import numpy as np

docs = [
    "The company announced a dividend.",
    "The company announced a new contract.",
    "The board approved the dividend payment.",
    "The company signed a contract with a bank.",
    "The company announced results, the company announced a new factory, and the company announced a dividend.",
]

docs.append(docs[0]*10)

query = "dividend payment"
stop = {"the", "a", "with", "and"}

In [ ]:
# creates a list from the input sentence, excluding the words in list called stop
def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [w for w in words if w not in stop]

In [ ]:
# 1) Split each sentence into tokens (lowercase words, stop words removed)
tokens = [tokenize(d) for d in docs]

# 2) Build the vocabulary: every distinct word, sorted alphabetically.
#    Each word becomes one column (one axis) of the vector space.
vocab = sorted({w for d in tokens for w in d})

#    idx maps each word to its column number, e.g. idx["dividend"] = 6
idx = {}
for j, w in enumerate(vocab):
    idx[w] = j

# 3) A[i, j] = number of times vocab word j appears in sentence i
#    (rows = sentences, columns = words)
A = np.zeros((len(docs), len(vocab)))
for i, d in enumerate(tokens):
    for w in d:
        A[i, idx[w]] += 1

# 4) q[j] = number of times vocab word j appears in the query
#    (same columns as A, so q can be multiplied with each row)
q = np.zeros(len(vocab))
for w in tokenize(query):
    if w in idx:
        q[idx[w]] += 1

print(vocab)
print(A)
print(q)

# 5) Aq: each entry is the dot product of one sentence (row of A) with q.
#    With word counts, this is how many query words ("dividend", "payment")
#    appear in that sentence.
#    Sentence 1: "dividend" once        -> 1
#    Sentence 3: both query words       -> 2
#    Sentence 5: "dividend" once        -> 1
#    Sentence 6: sentence 1 x 10        -> 10
print("Aq =", A @ q)

['announced', 'approved', 'bank', 'board', 'company', 'contract', 'dividend', 'factory', 'new', 'payment', 'results', 'signed']
[[ 1.  0.  0.  0.  1.  0.  1.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  1.  1.  0.  0.  1.  0.  0.  0.]
 [ 0.  1.  0.  1.  0.  0.  1.  0.  0.  1.  0.  0.]
 [ 0.  0.  1.  0.  1.  1.  0.  0.  0.  0.  0.  1.]
 [ 3.  0.  0.  0.  3.  0.  1.  1.  1.  0.  1.  0.]
 [10.  0.  0.  0. 10.  0. 10.  0.  0.  0.  0.  0.]]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0.]
Aq = [ 1.  0.  2.  0.  1. 10.]


In [4]:
# find the l2 norms of each sentence in A and q
row_norms = []
for i in range(len(A)):
    squared = 0
    for word in A[i]:
        squared += word**2
    row_norms.append(pow(squared, 0.5))

q_norm = 0
for i in q:
    q_norm += i**2
q_norm = pow(q_norm, 0.5)

print(row_norms)
print(q_norm)

[np.float64(1.7320508075688772), np.float64(2.0), np.float64(2.0), np.float64(2.0), np.float64(4.69041575982343), np.float64(17.320508075688775)]
1.4142135623730951


In [5]:
cos_similarities = []
for i in range(len(A)):
    sim = (A[i] @ q) / (row_norms[i] * q_norm)
    cos_similarities.append(sim)

# A vektöründeki her satırın q deki her satır ile olan cos benzerliği ölçülür
print(cos_similarities)

[np.float64(0.40824829046386296), np.float64(0.0), np.float64(0.7071067811865475), np.float64(0.0), np.float64(0.1507556722888818), np.float64(0.40824829046386296)]


## What the dot product means

The dot product can be written in terms of lengths and the angle between the vectors:

$$
a \cdot b = \|a\|\,\|b\|\,\cos\theta
$$

So it mixes two things: **how long** the vectors are and **how closely they
point in the same direction**.

### Our results

Query: "dividend payment". Sentence 6 is sentence 1 repeated 10 times
(`docs[0] * 10`), so its vector is exactly `10 × row 1`.

| # | Sentence | Dot product (row · q) | L2 norm of row | Cosine(row, q) |
|---|---|---|---|---|
| 1 | The company announced a dividend. | 1 | 1.73 | 0.41 |
| 2 | The company announced a new contract. | 0 | 2.00 | 0.00 |
| 3 | The board approved the dividend payment. | 2 | 2.00 | 0.71 |
| 4 | The company signed a contract with a bank. | 0 | 2.00 | 0.00 |
| 5 | The company announced results, ... a dividend. | 1 | 4.69 | 0.15 |
| 6 | Sentence 1 × 10 | 10 | 17.32 | 0.41 |

With word counts, the dot product counts the query words found in each
sentence. A score of 0 means the vectors are orthogonal: no shared words.

### What the dot product gets wrong

- **Sentences 1 and 5 tie at 1**, even though sentence 1 is about dividends
  and sentence 5 is mostly about results and a factory.
- **Sentence 6 jumps to the top with 10**, above sentence 3, which contains
  both query words. It adds no new information, it is just longer.

The dot product rewards length, not relevance.

### What cosine fixes

Cosine divides by the norms, so only direction matters:

- Sentence 1 (0.41) now ranks clearly above sentence 5 (0.15).
- Sentences 1 and 6 both score 0.41: same direction, different length.
  The factor of 10 cancels out:

$$
\cos(10a, q) = \frac{10\,(a \cdot q)}{10\,\|a\|\,\|q\|} = \cos(a, q)
$$

Final ranking by cosine: **3 > 1 = 6 > 5 > 2 = 4**